# Статистическая модель формы сердца

**Вопрос исследования.** Насколько компактное представление формы, построенное по автоматическим маскам других добровольцев, позволяет описать исходную форму сердца Ника? Дополнительно проверяется, выполняет ли установленный интерфейс MONAI Physio математически ожидаемое восстановление на синтетических поверхностях.

Это два вычислительных опыта с разным смыслом: первый оценивает сжатие реальных автоматических масок, второй проверяет программную реализацию метода главных компонент. Ни один из них не предсказывает движение сердца по одной КТ и не служит анатомической валидацией.

## Данные и разделение добровольцев

Для каждого из трёх добровольцев с псевдонимами Adam, Georg и Nix использована автоматическая маска всего сердца (`whole_heart`) из фазы 00 4D-КТ. Маски получены TotalSegmentator v2. Принятой независимой ручной разметки для этих сравнений нет.

В каждом разбиении два добровольца формируют базис, третий исключён из его построения. Основной проверочный человек — Ник. Его маска фазы 00 **используется при проекции на готовый базис**, то есть для нахождения коэффициента формы, и затем для расчёта Dice. Фазы 01 и далее в опыте не участвуют. Поэтому результат показывает, насколько хорошо базис описывает *известную* маску Ника; он не показывает, какой была бы неизвестная маска без КТ.

## Представление формы и контроль расчёта

Маску центрируют по центру её объёма и переносят на общую сетку 64 × 64 × 64 с шагом 3,5 мм. В каждом узле рассчитывают подписанное расстояние до границы в миллиметрах: совокупность этих значений образует вектор формы. Метод главных компонент (PCA) строит среднюю форму и направления её изменчивости по двум обучающим добровольцам. Для исключённого добровольца сравнивают среднюю форму без компонент и её проекцию на одну компоненту.

Dice равен удвоенному объёму пересечения двух бинарных масок, делённому на сумму их объёмов; значение 1 соответствует совпадению масок. Исходные маски проверяют на непустоту, бинарность и размещение в общей сетке. Разделение по псевдонимам и контрольные суммы опорных файлов фиксирует [исполняемый расчёт](mp05_07_eval.py). Центрирование исключает абсолютное положение сердца, а крупный шаг сетки сглаживает мелкие анатомические детали. Соответствие вершин для дальнейшего движения или FEM такой способ не создаёт.

## Сжатие автоматических масок

| Исключённый доброволец | Dice средней формы | Dice после проекции на одну компоненту | Изменение Dice |
|---|---:|---:|---:|
| Adam | 0,8625 | 0,8532 | −0,0094 |
| Georg | 0,7839 | 0,8250 | +0,0411 |
| Nix | 0,8095 | 0,8236 | +0,0141 |

Для Ника проекция повышает Dice на 0,0141 относительно средней формы. У Adam тот же приём ухудшает совпадение на 0,0094. При двух обучающих добровольцах в каждом разбиении можно получить лишь одну содержательную компоненту. Малый корпус, использование проверочной маски для проекции и отсутствие ручного анатомического референса не позволяют оценить точность прогноза для новых людей или выбрать оптимальную размерность модели.

## Проверка интерфейса MONAI Physio на синтетических формах

В отдельном тесте [скрипт](mp05_07_monai_pca_smoke.py) передал в `WorkflowCreateStatisticalModel.process()` пять обучающих и одну исключённую поверхность по 362 вершины. Все поверхности созданы из одного шаблона с двумя заданными гладкими модами в миллиметрах. Поэтому соответствие индексов вершин и граней известно заранее; этапы поиска соответствий через ICP и деформируемую регистрацию в этом тесте пропущены. Выполнялись этапы извлечения поверхности, PCA и формирования результата MONAI Physio версии 2026.09.1.

| Число компонент | Среднеквадратичная ошибка координат вершин исключённой формы, мм |
|---:|---:|
| 0 | 0,231853 |
| 1 | 0,221267 |
| 2 | 2,11 × 10⁻¹⁴ |

Доли объяснённой дисперсии двух компонент равны 0,664393 и 0,335607. Практически точное восстановление двумя компонентами ожидаемо, поскольку синтетические данные построены из двух мод. Этот результат подтверждает выполнение выбранного маршрута PCA API, но не точность на КТ и не качество регистрации реальных поверхностей.

## Вывод и воспроизведение

По маске Ника показано ограниченное улучшение описания **уже известной формы** одной компонентой PCA. Оно не проверяет точность автоматической сегментации и не даёт прогноза других фаз. Перед использованием геометрии в FEM потребуются проверка анатомической области, точности поверхности и влияния её изменения на электрический расчёт в одинаковой постановке.

Реальный опыт воспроизводится командой `python mp05_07_eval.py pca --data-root <локальный_корень_производных_4D_КТ>` при наличии `numpy`, `scipy` и `nibabel`. Синтетический опыт запускается скриптом `mp05_07_monai_pca_smoke.py`; его VTP и JSON остаются в исключённом из Git каталоге `runs/`. Происхождение и дополнительные численные результаты приведены в [протоколе опытов](mp05_07_results.md). В ноутбук не встроены исходные КТ или индивидуальные маски.